# Environment Check

The notebook confirms that this machine can run the exercises. It checks the Python version, imports every package the exercises use, and renders a page in the browser that the exercises drive. Run every cell in order. Each cell ends with a line that says what it confirmed, and a cell that fails raises an error that names the problem.

The notebook needs no internet access, because its one page is served from this machine.

## Python

The exercises run on Python 3.12. A different version means that the notebook is running on a kernel other than the `.venv` environment of this repository.

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), f"The kernel runs Python {sys.version.split()[0]}, and the exercises need Python 3.12."
print(f"Python {sys.version.split()[0]} runs from {sys.prefix}.")

## Packages

A package that fails to import is missing from the environment, and running `uv sync` in the repository root installs it.

In [ ]:
import importlib
from importlib.metadata import version

# Each module maps to the distribution that installs it, because the two names differ for some packages.
PACKAGES = {
    "bs4": "beautifulsoup4",
    "lxml": "lxml",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "playwright": "playwright",
    "requests": "requests",
    "scipy": "scipy",
    "seaborn": "seaborn",
    "statsmodels": "statsmodels",
}

for module, distribution in PACKAGES.items():
    importlib.import_module(module)
    print(f"{distribution:<16}{version(distribution)}")
print(f"All {len(PACKAGES)} packages import.")

## Browser

Exercise 1 collects a page that a script fills in after it loads, which takes a real browser. The cell serves such a page from this machine and reads it through Chromium. An error that mentions a missing executable means that the browser is not installed, and running `uv run playwright install chromium` in the repository root installs it.

In [ ]:
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

from playwright.async_api import async_playwright

# The paragraph is empty in the markup, so only a browser that runs the script sees the text.
PAGE = (
    b'<!doctype html><title>Check</title><p id="target"></p>'
    b'<script>document.getElementById("target").textContent = "rendered";</script>'
)


class Page(BaseHTTPRequestHandler):
    def do_GET(self) -> None:
        self.send_response(200)
        self.send_header("Content-Type", "text/html; charset=utf-8")
        self.send_header("Content-Length", str(len(PAGE)))
        self.end_headers()
        self.wfile.write(PAGE)

    def log_message(self, format: str, *args: object) -> None:
        pass


# Port 0 lets the system choose a free port.
server = ThreadingHTTPServer(("127.0.0.1", 0), Page)
threading.Thread(target=server.serve_forever, daemon=True).start()

async with async_playwright() as playwright:
    browser = await playwright.chromium.launch()
    tab = await browser.new_page()
    await tab.goto(f"http://127.0.0.1:{server.server_port}/")
    text = await tab.text_content("#target")
    await browser.close()
server.shutdown()

assert text == "rendered", f"Chromium returned {text!r} instead of the text the script writes."
print("Chromium rendered a page that a script fills in.")

## Result

If every cell above printed its confirmation, the machine is ready for the exercises.